# Modelo GARCH: Entrenamiento y Evaluación

In [1]:
import os
import random
import sys
import warnings
from pathlib import Path
import optuna

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.models.garch_wrapper import GARCHWrapper
from src.eval.scorer import VolatilityScorer
from src.trainer.walk_forward import WalkForwardTrainer
from src.utils import trainer_toolkit as tt
from src.utils import ts_toolkit as tst
from src.utils import test_toolkit as ts

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

OUT_PATH = PROJECT_ROOT / "data" / "processed"
RESULTS_PATH = PROJECT_ROOT / "results" / "GARCH"
MODELS_PATH = PROJECT_ROOT / "models"

RESULTS_PATH.mkdir(parents=True, exist_ok=True)
MODELS_PATH.mkdir(parents=True, exist_ok=True)

df_train = pd.read_parquet(OUT_PATH / "brent_train.parquet")
y_train = df_train["log_returns"].values.astype("float64")
rs_train = df_train["rs_daily"].values.astype("float64")

df_test = pd.read_parquet(OUT_PATH / "brent_test.parquet")
y_test = df_test["log_returns"].values.astype("float64")
rs_test = df_test["rs_daily"].values.astype("float64")

assert not np.isnan(y_train).any() and not np.isnan(rs_train).any(), "NaNs en train"
assert not np.isnan(y_test).any() and not np.isnan(rs_test).any(), "NaNs en test"
assert df_train.index.max() < df_test.index.min(), "Solapamiento temporal train/test"

print("=" * 70)
print(f"Train: {len(y_train)} obs  ({df_train.index[0].date()} -> {df_train.index[-1].date()})")
print(f"Test:  {len(y_test)} obs  ({df_test.index[0].date()} -> {df_test.index[-1].date()})")
print("=" * 70)

Train: 2171 obs  (2014-01-03 -> 2022-11-03)
Test:  953 obs  (2022-11-07 -> 2026-08-21)


:::{admonition} Resumen
:class: info
- El protocolo Walk-Forward seleccionó al GARCH(2,2) con distribución Normal como configuración óptima.
- La distribución $t$ de Student describió mejor las colas pesadas de los residuos, pero se empleó la Normal para garantizar la convergencia numérica del MLE.
- La estructura GARCH capturó la dependencia temporal del Brent, eliminando la necesidad de añadir rezagos excesivos y manteniendo la estacionariedad.
- El modelo dejó residuos fuera de muestra libres de autocorrelación, efectos ARCH remanentes y estructuras no lineales, indicando una correcta especificación.
- El modelo subestimó la magnitud de los shocks extremos de volatilidad, mostrando rigidez ante los cambios abruptos de régimen observados en el conjunto de prueba.
:::

## Introducción

El modelo GARCH (*Generalized Autoregressive Conditional Heteroskedasticity*) permite modelar la dinámica de la volatilidad en series financieras, como los retornos del petróleo Brent. Propuesto por Bollerslev (1986) como una extensión del modelo ARCH de Engle (1982), incorpora los choques pasados y los valores rezagados de la varianza condicional. 

El objetivo de esta sección es presentar, estimar y evaluar el modelo GARCH bajo el protocolo del benchmark, proporcionando la base conceptual para el análisis de modelos de volatilidad subsecuentes.

### Fundamentos

Los retornos logarítmicos diarios se representan como un proceso estocástico:

$$ r_t = \mu_t + \varepsilon_t, $$

donde $\mu_t$ denota la media condicional y $\varepsilon_t$ representa la innovación. Al asumir que la serie es estacionaria y su media está centrada en cero ($\mu_t = 0$), se tiene que $r_t = \varepsilon_t$. 

En su formulación clásica, las innovaciones siguen:

$$ \varepsilon_t = \sigma_t z_t, \quad z_t \sim i.i.d. \,\,\, \mathcal{N}(0,1), $$

con $\sigma_t^2$ denotando la varianza condicional y $z_t$ una variable aleatoria normal estándar. Bajo este modelo, la innovación $\varepsilon_t$ satisface:

$$ \mathbb{E}[\varepsilon_t \mid \mathcal{F}_{t-1}] = 0, \qquad \text{Var}(\varepsilon_t \mid \mathcal{F}_{t-1}) = \sigma_t^2, $$

es decir, la media condicional de la innovación es cero y su varianza condicional coincide con la varianza del proceso.

El supuesto de normalidad sobre $z_t$ puede relajarse. El análisis exploratorio de datos (EDA) evidenció que los retornos del petróleo Brent presentan colas pesadas y asimetría. Por esta razón, la distribución normal se sustituye por distribuciones como la Student-$t$ o la Generalized Error Distribution (GED). En todos los casos, la distribución de $z_t$ se mantiene estandarizada, con media cero y varianza unitaria, de modo que la varianza condicional de $\varepsilon_t$ sigue siendo $\sigma_t^2$.

### Modelo GARCH

El modelo GARCH generaliza el modelo ARCH incorporando rezagos de la varianza condicional. Su formulación general es:

$$
\sigma_t^2 =
\omega +
\sum_{i=1}^{q} \alpha_i \varepsilon_{t-i}^2 +
\sum_{j=1}^{p} \beta_j \sigma_{t-j}^2
$$

Donde:

- $\omega$: constante que representa el nivel basal de la varianza.
- $\alpha_i$: coeficientes que miden la influencia de los choques pasados al cuadrado.
- $\beta_j$: coeficientes que representan la persistencia de la varianza condicional.
- $q$: orden de los rezagos de las innovaciones al cuadrado.
- $p$: orden de los rezagos de la varianza condicional.

A diferencia del modelo ARCH, el GARCH representa la persistencia de la volatilidad con un menor número de parámetros al incorporar directamente los rezagos de la varianza condicional.

Para un modelo GARCH(1,1), la ecuación es:

$$
\sigma_t^2 =
\omega +
\alpha_1 \varepsilon_{t-1}^2 +
\beta_1 \sigma_{t-1}^2
$$

Bajo el supuesto de coeficientes no negativos ($\alpha_i \geq 0$, $\beta_j \geq 0$), la condición de estacionariedad de segundo orden es:

$$
\alpha_1 + \beta_1 < 1
$$

Esta condición garantiza que la varianza incondicional sea finita y que el impacto de los choques se disipe a largo plazo. De manera general, para un modelo GARCH($p$,$q$), la condición de estacionariedad exige que:

$$
\sum_{i=1}^{q} \alpha_i + \sum_{j=1}^{p} \beta_j < 1
$$

Bajo este supuesto, la varianza incondicional de largo plazo (el nivel medio al que tiende la serie) está dada por:

$$
\mathbb{E}[\sigma_t^2] = \frac{\omega}{1 - \sum_{i=1}^{q} \alpha_i - \sum_{j=1}^{p} \beta_j}
$$

La implementación computacional del modelo GARCH se realiza utilizando la librería `arch` de Python en su versión `8.0.0`.

## Entrenamiento

Para la selección de hiperparámetros, se ejecutaron cinco réplicas independientes bajo el esquema Walk-Forward. Se implementó optimización bayesiana mediante Optuna (algoritmo TPE) para explorar el espacio de configuraciones de forma eficiente. La búsqueda abarcó los órdenes de rezago $p$ y $q$ en el rango $\{1, 2, 3\}$, junto con tres distribuciones para las innovaciones: Normal, t de Student y GED. El desempeño se evaluó midiendo la pérdida QLIKE sobre los conjuntos de validación.

Para controlar el costo computacional, se fijó un límite de 50 evaluaciones (*trials*) por iteración, incorporando una parada temprana (*Early Stopping*) tras 15 intentos sin mejoras. Las configuraciones que fallaron en la convergencia de la Máxima Verosimilitud (MLE) recibieron una penalización en su QLIKE, asegurando que el algoritmo descartara sistemáticamente los modelos numéricamente inestables durante el proceso de búsqueda.

Con el fin de garantizar la estabilidad numérica del optimizador MLE, los retornos logarítmicos se multiplicaron por un factor de escala de 100 durante la estimación. Para preservar la consistencia, las varianzas pronosticadas por el modelo se dividieron por $100^2$, devolviéndolas a su escala original antes de ser evaluadas. De este modo, tanto la varianza realizada (Rogers-Satchell) como la varianza predicha se compararon en la misma escala.

In [3]:
def garch_search_space(trial: optuna.Trial) -> dict:
    return {
        "p": trial.suggest_int("p", 1, 3),
        "q": trial.suggest_int("q", 1, 3),
        "dist": trial.suggest_categorical("dist", ["normal", "t", "ged"]),
        "seed": 42
    }

scorer = VolatilityScorer()

for exec_id in range(1, 6):
    current_warmup = 1200 + (exec_id * 10)

    exec_dir = RESULTS_PATH / f"exec_{exec_id}"
    exec_dir.mkdir(parents=True, exist_ok=True)

    print(f"Iniciando GARCH | Ejecución {exec_id}/5 | warmup={current_warmup}")

    trainer = WalkForwardTrainer(
        model_class=GARCHWrapper,
        scorer=scorer,
        output_dir=exec_dir,
        n_splits=5,
        warmup=current_warmup,
        embargo=1,
        n_trials=50,                          
        suggest_params_fn=garch_search_space, 
        early_stop_rounds=15
    )

    trainer.run(y_train, rs_train)

Iniciando GARCH | Ejecución 1/5 | warmup=1210


  0%|          | 0/50 [00:00<?, ?it/s]

Iniciando GARCH | Ejecución 2/5 | warmup=1220


  0%|          | 0/50 [00:00<?, ?it/s]

Iniciando GARCH | Ejecución 3/5 | warmup=1230


  0%|          | 0/50 [00:00<?, ?it/s]

Iniciando GARCH | Ejecución 4/5 | warmup=1240


  0%|          | 0/50 [00:00<?, ?it/s]

Iniciando GARCH | Ejecución 5/5 | warmup=1250


  0%|          | 0/50 [00:00<?, ?it/s]

Respecto a la eficiencia del algoritmo de búsqueda, el mecanismo de parada temprana (*Early Stopping*) operó de manera óptima. Ninguna de las cinco réplicas alcanzó el límite máximo de 50 evaluaciones, deteniéndose entre el trial 18 y 28. El mejor desempeño se identificó durante las primeras iteraciones del algoritmo, lo que evidencia una rápida convergencia del espacio de hiperparámetros. Asimismo, el mecanismo de poda (*Pruning*) descartó eficientemente 6 modelos numéricamente inestables durante las primeras particiones, evitando el consumo innecesario de recursos. 

En términos de costo computacional, el entrenamiento completo bajo el esquema Walk-Forward (5 réplicas) totalizó 412 evaluaciones de folds exitosas. El tiempo de entrenamiento acumulado fue de 45.98 segundos, con un tiempo medio de 0.11 segundos por partición. Este bajo costo refleja la alta eficiencia computacional de los modelos paramétricos frente a los métodos iterativos, permitiendo su estimación y actualización diaria en entornos de producción sin requerir hardware especializado.

In [11]:
resumen_data = []
total_global_time = 0
total_global_folds = 0
global_best_qlike = float('inf')

for exec_id in range(1, 6):
    exec_dir = RESULTS_PATH / f"exec_{exec_id}"
    
    tiempo_total_rep = 0.0
    tiempo_promedio_fold = 0.0
    num_folds = 0
    total_trials, completos, podados, fallidos = 0, 0, 0, 0
    best_params_str = "-"
    mejor_qlike = "-"

    csv_files = list(exec_dir.glob("*_results.csv"))
    if csv_files:
        df_csv = pd.read_csv(csv_files[0])
        tiempo_total_rep = df_csv["train_time_sec"].sum()
        tiempo_promedio_fold = df_csv["train_time_sec"].mean()
        num_folds = len(df_csv)
        
    db_path = exec_dir / "optuna_study.db"
    if db_path.exists():
        try:
            storage_url = f"sqlite:///{db_path.resolve()}"
            study = optuna.load_study(study_name="GARCHWrapper_study", storage=storage_url)
            
            total_trials = len(study.trials)
            completos = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
            podados = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
            fallidos = len([t for t in study.trials if t.state == optuna.trial.TrialState.FAIL])
            
            mejor_qlike = round(study.best_value, 4)
            
            p_val = study.best_params.get('p', '?')
            q_val = study.best_params.get('q', '?')
            best_params_str = f"({p_val}, {q_val})"
            
            if mejor_qlike < global_best_qlike:
                global_best_qlike = mejor_qlike
                
        except Exception as e:
            print(f"Error leyendo BD en {exec_dir}: {e}")

    if num_folds > 0 or total_trials > 0:
        resumen_data.append({
            "Réplica": str(exec_id),
            "Total Trials": total_trials,
            "Completos": completos,
            "Podados": podados,
            "Mejor Config (p, q)": best_params_str,
            "Mejor QLIKE": mejor_qlike,
            "Tiempo Total (s)": round(tiempo_total_rep, 2),
            "Tiempo por Fold (s)": round(tiempo_promedio_fold, 3)
        })
        
        total_global_time += tiempo_total_rep
        total_global_folds += num_folds

resumen_data.append({
    "Réplica": "Total",
    "Total Trials": sum(r["Total Trials"] for r in resumen_data),
    "Completos": sum(r["Completos"] for r in resumen_data),
    "Podados": sum(r["Podados"] for r in resumen_data),
    "Mejor Config (p, q)": "--",
    "Mejor QLIKE": global_best_qlike,
    "Tiempo Total (s)": round(total_global_time, 2),
    "Tiempo por Fold (s)": round(total_global_time / total_global_folds, 3) if total_global_folds > 0 else 0
})

df_search_params = pd.DataFrame(resumen_data)
tt.display_styled_df(df=df_search_params, decimals=4)

Réplica,Total Trials,Completos,Podados,"Mejor Config (p, q)",Mejor QLIKE,Tiempo Total (s),Tiempo por Fold (s)
1,18,16,2,"(2, 3)",0.4353,9.1200,0.1320
2,24,23,1,"(2, 2)",0.4342,9.4300,0.1030
3,25,23,2,"(2, 2)",0.4335,8.9800,0.1070
4,28,27,1,"(2, 2)",0.4344,10.7500,0.1110
5,18,18,0,"(1, 3)",0.4344,7.7000,0.1100
Total,113,107,6,--,0.4335,45.9800,0.1120


Un análisis detallado de las réplicas revela que, aunque Optuna seleccionó configuraciones ligeramente distintas como la mejor opción en cada una (e.g., $p=2, q=2$ en tres réplicas, y $p=2, q=3$ o $p=1, q=3$ en las otras), la pérdida QLIKE se mantuvo estable en todo el rango (0.4335 a 0.4353). Esto demuestra la robustez del modelo GARCH ante variaciones paramétricas menores en la dinámica del Brent.

Para determinar la configuración definitiva del modelo, los resultados de las cinco réplicas se agregaron calculando la media y la desviación estándar de la pérdida QLIKE en cada fold.

In [22]:
tt.display_top_results(
    RESULTS_PATH, 
    top_n=5, 
    aggregate_mode="mean",
    column_names=(
        "p", "q", "Dist", 
        "Réplica 1", "Réplica 2", "Réplica 3", "Réplica 4", "Réplica 5", 
        "Global", "Fallos (%)"
    )
)

p,q,Dist,Réplica 1,Réplica 2,Réplica 3,Réplica 4,Réplica 5,Global,Fallos (%)
2,2,normal,-,0.434 ± 0.067,0.433 ± 0.072,0.434 ± 0.074,0.439 ± 0.084,0.435 ± 0.002,0.000
1,3,normal,0.435 ± 0.069,0.436 ± 0.076,0.435 ± 0.081,-,0.434 ± 0.083,0.435 ± 0.001,0.000
2,3,ged,0.435 ± 0.056,0.436 ± 0.063,-,0.436 ± 0.070,0.438 ± 0.078,0.436 ± 0.001,0.000
2,3,normal,-,0.437 ± 0.079,0.436 ± 0.084,0.439 ± 0.087,0.436 ± 0.086,0.437 ± 0.001,0.000
1,3,t,0.438 ± 0.060,0.438 ± 0.066,-,-,0.436 ± 0.072,0.437 ± 0.001,0.000


El análisis agregado reveló que las configuraciones con distribución Normal dominaron el ranking. Tanto el GARCH(2,2) como el GARCH(1,3) alcanzaron un QLIKE global de 0.435, con una desviación estándar mínima ($\pm 0.002$ y $\pm 0.001$, respectivamente) y una tasa de fallos del 0%. Con base en estos resultados, ambas configuraciones finalistas fueron seleccionadas para ser reentrenadas utilizando la muestra completa de entrenamiento.

In [2]:
best_model = GARCHWrapper(p=2, q=2, dist="normal")
best_model.fit(y_train, rs_train)

Tras el reentrenamiento y el análisis paramétrico, se seleccionó al GARCH(2,2) como el modelo final del benchmark por dos razones fundamentales. Primero, fue identificado como la mejor configuración en tres de las cinco réplicas independientes, demostrando una mayor consistencia algorítmica. Segundo, aunque el GARCH(1,3) presentó una dispersión ligeramente menor durante la validación, al ser reentrenado con la muestra completa evidenció problemas severos de multicolinealidad (términos de mayor orden colapsaron a cero con errores estándar inflados), indicando una estructura paramétrica inestable y sobreparametrizada. En contraste, el GARCH(2,2) mantuvo una estructura numéricamente estable.

Los coeficientes estimados y los intervalos de confianza (obtenidos mediante bootstrap por bloques) para el modelo GARCH(2,2) seleccionado se presentan a continuación:

In [14]:
df_params_boot = best_model.get_parameter_table(
    B=2000, decimals=4, block_size=22)

df_params_boot = df_params_boot.rename(columns={
    "Parameter": "Parámetro",
    "Coef": "Coeficiente",
    "Std. Error": "Error Estándar",
    "P-value": "p-valor",
    "CI Lower": "IC Inferior (95%)",
    "CI Upper": "IC Superior (95%)"
})

tt.display_styled_df(
    df=df_params_boot,
    decimals=4,
    p_value_cols=["p-valor"]
)

Parámetro,Coeficiente,Error Estándar,p-valor,IC Inferior (95%),IC Superior (95%)
omega,0.1661,0.0650,.011,0.1023,0.6516
alpha[1],0.1484,0.0481,.002,0.0000,0.1972
alpha[2],0.0538,0.0472,.255,0.0000,0.2037
beta[1],0.1437,0.1411,.308,0.0000,0.8901
beta[2],0.6444,0.1313,< .001,0.0000,0.8340


El análisis de los intervalos de confianza obtenidos mediante bootstrap por bloques (2000 iteraciones) muestra que el término constante $\omega$ presenta un intervalo estrictamente positivo, delimitado entre 0.1023 y 0.6710. Por otro lado, los intervalos para los parámetros de varianza ($\alpha_1, \alpha_2, \beta_1, \beta_2$) presentan un límite inferior que alcanza el valor de 0.0000. 

Este comportamiento se atribuye a dos factores. Primero, la restricción de no negatividad ($\alpha_i, \beta_j \ge 0$) impuesta por el optimizador MLE impide que los coeficientes adopten valores negativos durante el remuestreo. Segundo, la sobreparametrización del modelo GARCH(2,2) genera redundancia en la estructura de rezagos, lo que provoca que el optimizador desplace algunos coeficientes hacia la frontera inferior (cero) para maximizar la verosimilitud, mientras que $\beta_2$ (0.6444) absorbe la mayor parte de la persistencia.

A pesar de la amplitud de los intervalos, la condición de estacionariedad de segundo orden se verifica a partir de la suma de los coeficientes de persistencia:

$$
\sum_{i=1}^{q} \alpha_i + \sum_{j=1}^{p} \beta_j  = 0.9903 < 1
$$

Dado que el resultado es inferior a 1.0, la varianza incondicional es finita y los efectos de los choques se disipan en el largo plazo. Lo anterior confirma que, a pesar de la sobreparametrización y las restricciones numéricas, el modelo GARCH(2,2) mantiene una estructura estacionaria válida para el pronóstico de la varianza condicional. La proximidad de este valor al límite de la raíz unitaria evidencia una persistencia prolongada en la volatilidad del Brent. La vida media de los choques se estima en aproximadamente 71 días hábiles, lo que implica que la volatilidad requiere periodos extensos para retornar a su nivel basal tras un choque.

Para evaluar la estructura de los residuos sobre el conjunto de entrenamiento (In-Sample), se aplicaron pruebas de diagnóstico. Las pruebas de Ljung-Box (autocorrelación), ARCH-LM (heterocedasticidad) y BDS (dependencia no lineal) no rechazan la hipótesis nula (*p* > .05). Lo anterior indica que el modelo capturó la dependencia temporal y los efectos condicionales relevantes dentro de la muestra. No obstante, las pruebas de Jarque-Bera y Lilliefors rechazan la hipótesis nula de normalidad (*p* < .001). Esto evidencia que los residuos no se distribuyen de forma normal, un comportamiento consistente con la presencia de eventos extremos y la persistencia de la volatilidad, los cuales no logran ser absorbidos en su totalidad por el supuesto de normalidad.

In [3]:
std_residuals = best_model.model_fit.std_resid

df_diag = tt.run_model_diagnostics(
    std_residuals=std_residuals,
    lags=(10, 20, 30),
    p_adjust="bonferroni",
    alpha=0.05,
    english=False,
    column_titles=(
        "Prueba Estadística", "Estadístico",
        "p-valor", "p-adj", "Conclusión (α = 0.05)")
)

df_display = df_diag.copy()

df_display["Estadístico"] = df_display["Estadístico"].apply(
    lambda x: f"{x:.4f}" if pd.notna(x) else "--")

def _format_p(x):
    if pd.isna(x): return "--"
    if x < 0.001: return "< .001"
    formatted = f"{x:.3f}"
    return formatted[1:] if formatted.startswith("0.") else formatted

df_display["p-valor"] = df_display["p-valor"].apply(_format_p)
df_display["p-adj"] = df_display["p-adj"].apply(_format_p)

tt.display_styled_df(df=df_display, p_value_cols=[])

Prueba Estadística,Estadístico,p-valor,p-adj,Conclusión (α = 0.05)
Ljung-Box (Lag 10),7.7894,.649,1.000,Sin Autocorrelación
Ljung-Box (Lag 20),25.3687,.188,.563,Sin Autocorrelación
Ljung-Box (Lag 30),35.9194,.211,.632,Sin Autocorrelación
ARCH-LM (Lag 10),2.2262,.994,1.000,Sin Efectos ARCH
ARCH-LM (Lag 20),6.4301,.998,1.000,Sin Efectos ARCH
ARCH-LM (Lag 30),18.0074,.958,1.000,Sin Efectos ARCH
Jarque-Bera,2027.0840,< .001,--,Residuos No Normales
Lilliefors,0.0535,< .001,--,Residuos No Normales
BDS (Dim 2),-0.7985,.425,--,Sin Estructura No Lineal


El análisis exploratorio de datos (EDA) evidenció previamente la presencia de colas pesadas en los retornos del Brent. Dado que el modelo GARCH estimado asume Normalidad, se planteó la hipótesis de que los residuos estandarizados conservarían esta estructura de riesgo extremo no capturado. Para contrastarlo, se ajustó una distribución $t$ de Student a los residuos y se evaluó su correspondencia mediante un gráfico Q-Q, el cual compara los cuantiles empíricos con los cuantiles teóricos de la distribución propuesta.

En el panel derecho de la figura, se observa que los cuantiles empíricos presentan una alineación precisa con la línea de referencia diagonal en el centro de la distribución. Sin embargo, en los extremos (colas), los puntos se desvían de la recta teórica, indicando que incluso la distribución $t$ de Student no logra capturar en su totalidad los eventos más extremos. A pesar de estas desviaciones visuales en las colas, las pruebas formales de bondad de ajuste (Kolmogorov-Smirnov: $p = .5835$; Cramér-von Mises: $p = .4386$) no rechazan la hipótesis de pertenencia a la distribución $t$. Esto implica que, si bien la $t$ describe adecuadamente el cuerpo de los residuos, persisten componentes de riesgo extremo que el modelo GARCH paramétrico no logra absorber.

In [6]:
_, _ = tst.plot_distribution_fit(
    pd.Series(std_residuals),
    dist="t",
    titles=("Distribución de Residuos", "Gráfico Q-Q (t-Student)"),
    xlab="Residuo estandarizado",
    ylab_left="Densidad",
    xlab_right="Cuantiles teóricos",
    ylab_right="Cuantiles muestrales",
    ylim_left=(0, 0.7),
    legend_loc="upper left"
)

Para formalizar la comparación estadística entre ambas distribuciones, se realizó una prueba de Razón de Verosimilitud (LRT) junto con los criterios de información AIC y BIC. Los resultados, presentados a continuación, evidencian que la distribución $t$ de Student proporciona un ajuste superior. 

In [4]:
_= tst.distribution_fit_table(
    pd.Series(std_residuals),
    column_titles=(
        "Distribución", 
        "Log-Likelihood", 
        "AIC", 
        "BIC", 
        "Estadístico LRT", 
        "p-valor LRT"
    )
)

Distribución,Log-Likelihood,AIC,BIC,Estadístico LRT,p-valor LRT
Normal,"-3,079.22","6,162.45","6,173.81",-,-
t-Student,"-2,962.57","5,931.15","5,948.20",233.30,< .001


El estadístico LRT rechaza la hipótesis nula de igualdad de ajuste ($p < .001$). Los criterios de información evidencian una reducción de $\Delta\text{AIC} = 231.30$ y $\Delta\text{BIC} = 225.61$ a favor de la distribución $t$. Esto confirma que la inclusión del parámetro de grados de libertad ($\nu \approx 4.83$) mejora significativamente la descripción de los residuos sin penalizar en exceso la complejidad del modelo.

A pesar de que la $t$ de Student describe mejor la distribución empírica de los residuos, la mejor configuración seleccionada por Optuna empleó la distribución Normal. Este contraste responde a que la estimación del parámetro de grados de libertad de la $t$ mediante el optimizador MLE presentó inestabilidad numérica y fallos de convergencia durante el reentrenamiento diario del Walk-Forward. La distribución Normal, aunque teóricamente inferior para modelar las colas, garantizó una actualización diaria estable del modelo. En un entorno dinámico de pronóstico, esta estabilidad computacional resultó determinante para obtener una menor pérdida QLIKE media, demostrando que el desempeño predictivo empírico supera a la bondad de ajuste distribucional estática.

El análisis del ajuste sobre la muestra de entrenamiento mostró que la volatilidad condicional estimada siguió la trayectoria general de los clústeres de volatilidad. No obstante, el recuadro de zoom (crisis COVID-19) evidenció que el modelo subestimó la magnitud del pico máximo de volatilidad. Este resultado indicó que, incluso al procesar datos del propio conjunto de entrenamiento, el modelo presentó limitaciones para adaptarse a la magnitud de los shocks extremos.

In [5]:
rs_volatilidad = np.sqrt(rs_train)
pred_volatilidad = best_model.model_fit.conditional_volatility / 100.0
mask_crisis = (df_train.index >= "2020-01-01") & (df_train.index <= "2020-06-30")

ts.plot_volatility_fit(
    dates=df_train.index,
    rs_variance=rs_volatilidad,      
    pred_variance=pred_volatilidad,
    title="Ajuste en la Muestra de Entrenaminento(GARCH 2,2)", ylab="Volatilidad",              
    label_realized=r"Volatilidad Realizada $\left(\sigma_R\right)$", 
    label_pred=r"Volatilidad GARCH $\left(\hat{\sigma}_t\right)$",
    ylim=(-0.01, 0.35), legend_bbox=(0.01, 0.98), 
    zoom_slice=mask_crisis, zoom_pad_frac=0.04,
    zoom_title="Zoom: Crisis COVID-19",
    zoom_bbox=(0.18, 0.35, 0.40, 0.40) 
)

## Evaluación del Modelo

La evaluación del rendimiento de un modelo de volatilidad requirió contrastar su capacidad de ajuste sobre los datos observados con su capacidad predictiva sobre datos no vistos. Mientras que el conjunto de entrenamiento (In-Sample) permitió verificar la convergencia y el ajuste del algoritmo a la dinámica histórica, la evaluación Out-of-Sample (conjunto de prueba) determinó la utilidad real del modelo ante nuevos regímenes de mercado. 

Para diagnosticar la presencia de sobreajuste (*overfitting*) o subajuste (*underfitting*), se aplicaron los criterios pre-registrados en el protocolo experimental. Estos criterios se basaron en la diferencia absoluta entre las métricas de prueba y entrenamiento, denotada como:

$$
\Delta_{abs} = M_{test} - M_{train}
$$

Se estableció un umbral de tolerancia absoluto uniforme de $\Delta_{abs} = 0.05$ para todas las métricas de evaluación. Un valor de $\Delta_{abs}$ que superara estos márgenes evidenció sobreajuste y falta de generalización.

Las predicciones fuera de muestra se obtuvieron ejecutando el método `forecast` del wrapper del modelo. Este procedimiento aplicó un esquema de ventana expansiva (*rolling forward*) para garantizar la ausencia de fuga de información, generando las varianzas condicionales predichas en la escala original para permitir su comparación directa con la varianza realizada.

In [9]:
preds_test = best_model.forecast(y_seq=y_test, rs_seq=rs_test)

df_preds = pd.DataFrame({
    "date": df_test.index, "rs_true": rs_test, "y_pred": preds_test})

output_path = RESULTS_PATH / "best_garch_predictions.csv"
df_preds.to_csv(output_path, index=False)

A partir de las varianzas predichas y los retornos reales del conjunto de prueba, se construyeron los residuos estandarizados fuera de muestra mediante la siguiente expresión:

$$
z_t = \frac{y_t}{\sqrt{\hat{\sigma}_t^2}}
$$

Los resultados del diagnóstico fueron consistentes con el diagnóstico In-Sample. Las pruebas estructurales (Ljung-Box, ARCH-LM y BDS) no rechazaron sus hipótesis nulas, lo que indicó que el modelo capturó la dinámica temporal sin dejar estructura remanente fuera de muestra. No obstante, las pruebas de normalidad rechazaron la hipótesis nula. Esto implicó que el supuesto paramétrico Normal no logró absorber las colas pesadas de los errores. Dicho hallazgo justificó empíricamente el uso de la pérdida QLIKE y motivó la exploración de arquitecturas con mayor flexibilidad distribucional.

In [25]:
rs_test_vol = df_preds["rs_true"]
pred_test_vol = df_preds["y_pred"]

std_residuals_oos = y_test / np.sqrt(df_preds["y_pred"])
std_residuals_oos = std_residuals_oos[np.isfinite(std_residuals_oos)]

df_diag = tt.run_model_diagnostics(
    std_residuals=std_residuals_oos,
    lags=(10, 20, 30),
    p_adjust="bonferroni",
    alpha=0.05,
    english=False,
    column_titles=(
        "Prueba Estadística", "Estadístico",
        "p-valor", "p-adj", "Conclusión (α = 0.05)")
)
df_display = df_diag.copy()

df_display["Estadístico"] = df_display["Estadístico"].apply(
    lambda x: f"{x:.4f}" if pd.notna(x) else "--")

def _format_p(x):
    if pd.isna(x): return "--"
    if x < 0.001: return "< .001"
    formatted = f"{x:.3f}"
    return formatted[1:] if formatted.startswith("0.") else formatted

df_display["p-valor"] = df_display["p-valor"].apply(_format_p)
df_display["p-adj"] = df_display["p-adj"].apply(_format_p)

tt.display_styled_df(df=df_display, p_value_cols=[])

Prueba Estadística,Estadístico,p-valor,p-adj,Conclusión (α = 0.05)
Ljung-Box (Lag 10),10.9111,.364,1.000,Sin Autocorrelación
Ljung-Box (Lag 20),25.0901,.198,.594,Sin Autocorrelación
Ljung-Box (Lag 30),28.2156,.559,1.000,Sin Autocorrelación
ARCH-LM (Lag 10),5.5324,.853,1.000,Sin Efectos ARCH
ARCH-LM (Lag 20),10.6502,.955,1.000,Sin Efectos ARCH
ARCH-LM (Lag 30),17.6025,.965,1.000,Sin Efectos ARCH
Jarque-Bera,119.2357,< .001,--,Residuos No Normales
Lilliefors,0.0418,< .001,--,Residuos No Normales
BDS (Dim 2),-1.4307,.153,--,Sin Estructura No Lineal


El análisis distribucional de los residuos fuera de muestra indicó un ajuste a la distribución $t$ de Student. Las pruebas de bondad de ajuste no rechazaron la hipótesis de pertenencia a dicha distribución (Kolmogorov-Smirnov: $p = 0.7624$; Cramér-von Mises: $p = 0.5622$), mientras que la prueba de Razón de Verosimilitud evidenció una mejora frente al supuesto de Normalidad ($ \text{LRT} = 38.26, p < .001$). El parámetro de grados de libertad estimado fue $\nu \approx 6.98$, lo cual se reflejó en una menor concentración de la densidad en el centro (pico más achatado y colas más ligeras) y en un ajuste de las colas en el gráfico Q-Q superior al observado en el entrenamiento.

In [29]:
_, _ = tst.plot_distribution_fit(
    pd.Series(std_residuals_oos),
    dist="t",
    titles=("Distribución de Residuos", "Gráfico Q-Q (t-Student)"),
    xlab="Residuo estandarizado",
    ylab_left="Densidad",
    xlab_right="Cuantiles teóricos",
    ylab_right="Cuantiles muestrales",
    ylim_left=(0, 0.7),
    legend_loc="upper left"
)

Este comportamiento distribucional se atribuyó a las diferencias en la composición de los conjuntos de datos. Mientras que el conjunto de entrenamiento incluyó eventos de volatilidad extrema y prolongada (como la crisis del COVID-19 en 2020 y el inicio del conflicto en Ucrania en 2022), el conjunto de prueba presentó shocks de menor duración e intensidad relativa. Al carecer de eventos tan disruptivos, la distribución de los residuos de prueba presentó colas más ligeras. Esto permitió que la distribución $t$ describiera el riesgo de cola de manera más precisa, confirmando que la bondad de ajuste distribucional depende en gran medida de la severidad de los regímenes de volatilidad presentes en cada muestra.

El modelo GARCH(2,2) cumplió con los criterios de adecuación pre-fijados para la mayoría de las métricas. La pérdida principal QLIKE se mantuvo por debajo del umbral de tolerancia absoluto establecido. De igual forma, las métricas de error puntual (MSE, RMSE, MAE) y probabilísticas (CRPS y Pinball) presentaron variaciones marginales, lo cual apoyo la capacidad predictiva del modelo fuera de la muestra de entrenamiento. 

In [28]:
preds_train_var = (best_model.model_fit.conditional_volatility ** 2) / 10000.0
preds_test_var = df_preds["y_pred"].values

ts.display_train_test_comparison(
    rs_train=rs_train,
    preds_train=preds_train_var,
    rs_test=rs_test,
    preds_test=preds_test_var,
    column_titles=(
        "Métrica", 
        "Entrenamiento", 
        "Prueba", 
        "Diferencia"
    )
)

Métrica,Entrenamiento,Prueba,Diferencia
QLIKE,0.3843,0.4103,+0.0260
CRPS,0.0126,0.0123,-0.0003
Pinball (τ=0.05),0.0030,0.0028,-0.0001
Pinball (τ=0.95),0.0014,0.0016,+0.0002
MSE,0.0000,0.0000,+0.0000
RMSE,0.0018,0.0022,+0.0004
MAE,0.0005,0.0005,-0.0000
HMSE,0.9898,2.5725,+1.5828
HMAE,0.6226,0.6727,+0.0501


Las métricas ajustadas por heterocedasticidad (HMSE y HMAE) presentaron incrementos que superaron el umbral establecido. Este comportamiento se explicó por la naturaleza de dichos indicadores: el HMSE pondera los errores de predicción por el inverso de la volatilidad, penalizando severamente las subestimaciones durante periodos de alta volatilidad. Dado que el modelo presentó dificultades para capturar la magnitud del pico extremo observado en 2026, el HMSE reflejó dicha limitación ante eventos extremos. Lo anterior no indicó un sobreajuste global de la dinámica de la varianza, sino que apuntó hacia una sensibilidad específica a los shocks abruptos.

In [22]:
rs_test_vol = np.sqrt(df_preds["rs_true"])
pred_test_vol = np.sqrt(df_preds["y_pred"])
mask_crisis = (df_preds["date"] >= "2026-02-01") & (df_preds["date"] <= "2026-06-30")

ts.plot_volatility_fit(
    dates=df_preds["date"],
    rs_variance=rs_test_vol,      
    pred_variance=pred_test_vol,
    title="Ajuste en fuera de Muestra (GARCH 2,2)", 
    ylab="Volatilidad",              
    label_realized=r"Volatilidad Realizada $\left(\sigma_R\right)$", 
    label_pred=r"Volatilidad GARCH $\left(\hat{\sigma}_t\right)$",
    
    ylim=(-0.01, 0.35), 
    legend_bbox=(0.01, 0.98), 
    
    zoom_slice=mask_crisis, zoom_pad_frac=0.04,
    zoom_title="Zoom: Pico de Volatilidad (2026)",
    zoom_bbox=(0.25, 0.35, 0.40, 0.40) 
)

## Conclusiones

La implementación y evaluación del modelo GARCH(2,2) permitió establecer un punto de partida paramétrico para el pronóstico de la volatilidad del Brent. Los diagnósticos sobre los residuos fuera de muestra indicaron que el modelo logró capturar la dependencia temporal y la heterocedasticidad condicional, al no dejar estructura lineal ni efectos ARCH remanentes. Los criterios de adecuación y la estabilidad del QLIKE sugirieron que el modelo generalizó su capacidad predictiva sin incurrir en un sobreajuste severo.

El análisis de los resultados expuso limitaciones específicas de la especificación. Aunque el supuesto de Normalidad empleado durante la estimación no logró absorber el riesgo de cola, el análisis distribucional de los residuos de prueba evidenció colas más ligeras en comparación con el entrenamiento. El incremento en las métricas ajustadas por heterocedasticidad (HMSE y HMAE) reflejó la dificultad del modelo para adaptarse a la magnitud de shocks estructurales extremos. Esta rigidez sugirió que, si bien el GARCH capturó la dinámica condicional habitual, presentó limitaciones al enfrentarse a cambios abruptos de régimen.

Estos hallazgos motivaron la transición hacia modelos con mayor flexibilidad. La evidencia empírica de esta sección sustentó la necesidad de evaluar modelos que capturan explícitamente la memoria larga y la heterogeneidad de los agentes (como el modelo HAR) y arquitecturas no paramétricas capaces de aprender relaciones no lineales complejas (como las redes neuronales LSTM y GRU).